# Demo: GNN–BERT Music Context Understanding

This notebook demonstrates end-to-end inference for the GNN–BERT Music Context Understanding system.

**Pipeline:**
1. Load audio features → Build graph
2. Generate text description
3. Predict tags + emotion (valence/arousal)
4. Retrieve matching captions
5. Visualize results

In [ ]:
import sys
sys.path.append('..')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import yaml
import zipfile
from pathlib import Path

# Project imports
from src.audio_features import AudioFeatureExtractor
from src.graph_builder import MusicGraphBuilder
from src.tag_generator import TagGenerator
from src.bert_encoder import BERTTagClassifier
from src.gnn_model import GraphSAGEEncoder, GNNClassifier
from src.fusion_model import GNNBERTFusionModel, CrossAttentionFusion
from src.contrastive import ContrastiveDualEncoder, compute_retrieval_metrics

from transformers import BertTokenizer

# Load config
with open('../config.yaml') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print('All modules loaded successfully!')

## Step 1: Load Audio Features & Build Graph

In [ ]:
# Extract features if needed
DATA_DIR = '../data/raw'
FEATURES_DIR = os.path.join(DATA_DIR, 'features')

if not os.path.exists(FEATURES_DIR):
    print('Extracting features...')
    with zipfile.ZipFile(os.path.join('..', '..', 'features.zip'), 'r') as z:
        z.extractall(DATA_DIR)

# Pick a sample song
feature_files = sorted([f for f in os.listdir(FEATURES_DIR) if f.endswith('.csv')])
sample_file = feature_files[0]
song_id = int(sample_file.replace('.csv', ''))
print(f'Selected song ID: {song_id}')

# Load OpenSMILE features
extractor = AudioFeatureExtractor(config)
raw_features = extractor.load_opensmile_features(os.path.join(FEATURES_DIR, sample_file))
print(f'Raw feature shape: {raw_features.shape} (frames x features)')

# Segment features (4 frames = 2 seconds per segment)
segments = extractor.segment_opensmile_features(raw_features)
print(f'Number of segments: {len(segments)}')
print(f'Segment feature dim: {segments[0].shape}')

In [ ]:
# Build graph from segments
graph_builder = MusicGraphBuilder(config)
segment_array = np.array(segments)
graph = graph_builder.build_segment_graph(segment_array)

print(f'Graph constructed:')
print(f'  Nodes: {graph.num_nodes}')
print(f'  Edges: {graph.edge_index.shape[1]}')
print(f'  Node feature dim: {graph.x.shape[1]}')
print(f'  Average degree: {graph.edge_index.shape[1] / graph.num_nodes:.2f}')

In [ ]:
# Visualize graph structure
import networkx as nx

edge_index = graph.edge_index.numpy()
G = nx.Graph()
G.add_nodes_from(range(graph.num_nodes))
for i in range(edge_index.shape[1]):
    G.add_edge(edge_index[0, i], edge_index[1, i])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Graph visualization
pos = nx.spring_layout(G, seed=42, k=2/np.sqrt(graph.num_nodes))
node_colors = np.arange(graph.num_nodes)  # Color by temporal position
nx.draw(G, pos, ax=axes[0], node_color=node_colors, cmap='viridis',
        node_size=100, edge_color='gray', alpha=0.7, with_labels=True, font_size=6)
axes[0].set_title(f'Music Segment Graph (Song {song_id})', fontsize=14)

# Degree distribution
degrees = [G.degree(n) for n in G.nodes()]
axes[1].hist(degrees, bins=range(max(degrees)+2), color='#3498db', 
             edgecolor='white', alpha=0.8, align='left')
axes[1].set_xlabel('Node Degree')
axes[1].set_ylabel('Count')
axes[1].set_title('Degree Distribution')

plt.tight_layout()
plt.savefig('../results/plots/demo_graph.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 2: Generate Text Description

In [ ]:
# Load annotations and generate tags
tag_gen = TagGenerator(config)
tag_data = tag_gen.generate_all()

# Get info for our sample song
if str(song_id) in tag_data:
    song_info = tag_data[str(song_id)]
elif song_id in tag_data:
    song_info = tag_data[song_id]
else:
    # Find closest song
    available = list(tag_data.keys())
    song_id_str = available[0]
    song_info = tag_data[song_id_str]
    print(f'Song {song_id} not found in annotations, using song {song_id_str}')

print(f'Song {song_id} Analysis:')
print(f'  Valence: {song_info["valence"]:.2f} / 9.0')
print(f'  Arousal: {song_info["arousal"]:.2f} / 9.0')
print(f'  Mood Tags: {song_info["mood_tags"]}')
print(f'  Genre Tags: {song_info["genre_tags"]}')
print(f'  Generated Description:')
print(f'    "{song_info["text_description"]}"')

## Step 3: Task 1 — BERT Tag Prediction

In [ ]:
# Initialize BERT model
tag_vocab = tag_gen.get_tag_vocabulary()
num_tags = len(tag_vocab)

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BERTTagClassifier(
    model_name='bert-base-uncased',
    num_tags=num_tags,
    freeze_layers=8
).to(device)

# Load trained weights if available
model_path = '../results/bert_tag_classifier.pt'
if os.path.exists(model_path):
    bert_model.load_state_dict(torch.load(model_path, map_location=device))
    print('Loaded trained BERT model')
else:
    print('Using untrained BERT model (run training first for accurate predictions)')

bert_model.eval()

# Predict tags from text description
text = song_info['text_description']
encoding = tokenizer(text, return_tensors='pt', max_length=128,
                     padding='max_length', truncation=True)
input_ids = encoding['input_ids'].to(device)
attention_mask = encoding['attention_mask'].to(device)

with torch.no_grad():
    logits, cls_emb, _ = bert_model(input_ids, attention_mask)
    probs = torch.sigmoid(logits).cpu().numpy()[0]

# Display predictions
print(f'\nBERT Tag Predictions for Song {song_id}:')
print('-' * 50)
for tag, prob in sorted(zip(tag_vocab, probs), key=lambda x: x[1], reverse=True):
    marker = '✓' if prob > 0.5 else ' '
    bar = '█' * int(prob * 30)
    print(f'  [{marker}] {tag:15s} {prob:.3f} {bar}')

## Step 4: Task 2 — GNN Classification

In [ ]:
# Initialize GNN model
gnn_encoder = GraphSAGEEncoder(
    in_channels=config['graph']['node_feature_dim'],
    hidden_channels=config['gnn']['hidden_dims'],
    dropout=config['gnn']['dropout']
)
gnn_classifier = GNNClassifier(
    encoder=gnn_encoder,
    graph_dim=config['gnn']['hidden_dims'][-1],
    num_classes=num_tags
).to(device)

model_path = '../results/gnn_classifier.pt'
if os.path.exists(model_path):
    gnn_classifier.load_state_dict(torch.load(model_path, map_location=device))
    print('Loaded trained GNN model')
else:
    print('Using untrained GNN model (run training first for accurate predictions)')

gnn_classifier.eval()

# Predict from graph
from torch_geometric.data import Batch
graph_batch = Batch.from_data_list([graph.to(device)])

with torch.no_grad():
    logits, graph_emb = gnn_classifier(graph_batch)
    probs_gnn = torch.sigmoid(logits).cpu().numpy()[0]

print(f'\nGNN Tag Predictions for Song {song_id}:')
print('-' * 50)
for tag, prob in sorted(zip(tag_vocab, probs_gnn), key=lambda x: x[1], reverse=True):
    marker = '✓' if prob > 0.5 else ' '
    bar = '█' * int(prob * 30)
    print(f'  [{marker}] {tag:15s} {prob:.3f} {bar}')

## Step 5: Task 3 — GNN–BERT Fusion

In [ ]:
# Initialize fusion model
fusion_model = GNNBERTFusionModel(
    gnn_encoder=GraphSAGEEncoder(
        in_channels=config['graph']['node_feature_dim'],
        hidden_channels=config['gnn']['hidden_dims'],
        dropout=config['gnn']['dropout']
    ),
    bert_encoder=BERTTagClassifier(
        model_name='bert-base-uncased',
        num_tags=num_tags,
        freeze_layers=8
    ),
    fusion_type='cross_attention',
    graph_dim=config['gnn']['hidden_dims'][-1],
    num_tags=num_tags,
    fused_dim=config['fusion']['fused_dim'],
    num_heads=config['fusion']['cross_attention_heads']
).to(device)

model_path = '../results/fusion_model.pt'
if os.path.exists(model_path):
    fusion_model.load_state_dict(torch.load(model_path, map_location=device))
    print('Loaded trained fusion model')
else:
    print('Using untrained fusion model (run training first for accurate predictions)')

fusion_model.eval()

# Predict
with torch.no_grad():
    tag_logits, val_pred, aro_pred, fused_emb, attn_weights = fusion_model(
        graph_batch, input_ids, attention_mask
    )
    probs_fusion = torch.sigmoid(tag_logits).cpu().numpy()[0]
    valence_pred = val_pred.cpu().item()
    arousal_pred = aro_pred.cpu().item()

print(f'\nGNN–BERT Fusion Predictions for Song {song_id}:')
print('-' * 50)
print(f'  Predicted Valence: {valence_pred:.2f} (Ground Truth: {song_info["valence"]:.2f})')
print(f'  Predicted Arousal: {arousal_pred:.2f} (Ground Truth: {song_info["arousal"]:.2f})')
print(f'\n  Tag Predictions:')
for tag, prob in sorted(zip(tag_vocab, probs_fusion), key=lambda x: x[1], reverse=True):
    marker = '✓' if prob > 0.5 else ' '
    bar = '█' * int(prob * 30)
    print(f'  [{marker}] {tag:15s} {prob:.3f} {bar}')

In [ ]:
# Visualize attention weights
if attn_weights is not None:
    attn = attn_weights.cpu().numpy()[0]  # (num_heads, 1, seq_len)
    
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].cpu())
    # Get actual tokens (non-padding)
    mask = attention_mask[0].cpu().numpy().astype(bool)
    actual_tokens = [t for t, m in zip(tokens, mask) if m]
    
    fig, axes = plt.subplots(1, min(attn.shape[0], 4), figsize=(16, 3))
    if not isinstance(axes, np.ndarray):
        axes = [axes]
    
    for head_idx, ax in enumerate(axes):
        if head_idx >= attn.shape[0]:
            break
        head_attn = attn[head_idx, 0, :len(actual_tokens)]
        ax.barh(range(len(actual_tokens)), head_attn, color='#3498db')
        ax.set_yticks(range(len(actual_tokens)))
        ax.set_yticklabels(actual_tokens, fontsize=7)
        ax.set_title(f'Head {head_idx + 1}', fontsize=11)
        ax.invert_yaxis()
    
    plt.suptitle('Cross-Attention Weights (Graph → Text)', fontsize=14)
    plt.tight_layout()
    plt.savefig('../results/plots/demo_attention.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Attention weights not available (fusion type may not be cross_attention)')

## Step 6: Model Comparison

In [ ]:
# Compare predictions across models
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(num_tags)
width = 0.25

bars1 = ax.bar(x - width, probs, width, label='BERT-only', color='#3498db', alpha=0.8)
bars2 = ax.bar(x, probs_gnn, width, label='GNN-only', color='#2ecc71', alpha=0.8)
bars3 = ax.bar(x + width, probs_fusion, width, label='GNN–BERT Fusion', color='#e74c3c', alpha=0.8)

# Ground truth markers
gt_tags = song_info['mood_tags'] + song_info['genre_tags']
for i, tag in enumerate(tag_vocab):
    if tag in gt_tags:
        ax.scatter(i, 1.05, marker='*', color='gold', s=200, zorder=5)

ax.set_xlabel('Tags', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title(f'Tag Predictions Comparison — Song {song_id}', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(tag_vocab, rotation=45, ha='right')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Threshold')
ax.legend(loc='upper right')
ax.set_ylim(0, 1.15)

plt.tight_layout()
plt.savefig('../results/plots/demo_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7: Case Studies

### Case Study 1: High Valence, High Arousal (Happy/Energetic)

In [ ]:
# Run 3 case studies across different V/A quadrants
case_studies = []

for song_key, info in tag_data.items():
    v, a = info['valence'], info['arousal']
    # Case 1: High V, High A
    if v > 6.5 and a > 6.5 and len(case_studies) == 0:
        case_studies.append(('High V / High A (Happy/Energetic)', song_key, info))
    # Case 2: Low V, High A
    elif v < 3.5 and a > 5.5 and len(case_studies) == 1:
        case_studies.append(('Low V / High A (Angry/Tense)', song_key, info))
    # Case 3: High V, Low A
    elif v > 5.5 and a < 3.5 and len(case_studies) == 2:
        case_studies.append(('High V / Low A (Relaxed/Peaceful)', song_key, info))
    if len(case_studies) == 3:
        break

for label, sid, info in case_studies:
    print(f'\n{"="*60}')
    print(f'Case Study: {label}')
    print(f'Song ID: {sid}')
    print(f'Valence: {info["valence"]:.2f}, Arousal: {info["arousal"]:.2f}')
    print(f'Mood Tags: {info["mood_tags"]}')
    print(f'Genre Tags: {info["genre_tags"]}')
    print(f'Description: {info["text_description"]}')
    print(f'{"="*60}')

## Step 8: Load and Display Final Results

If training has been completed, load and display the final comparison table.

In [ ]:
# Load results if available
results_path = '../results/metrics.json'
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    
    print('\n' + '='*70)
    print('FINAL RESULTS COMPARISON TABLE')
    print('='*70)
    print(f'{"Model":<25} {"Macro-F1":>10} {"AUC-PR":>10} {"MAE(emo)":>10} {"R@5":>10}')
    print('-'*70)
    
    for model_name, metrics in results.items():
        macro_f1 = f"{metrics.get('macro_f1', '-'):.4f}" if isinstance(metrics.get('macro_f1'), float) else '-'
        auc_pr = f"{metrics.get('auc_pr', '-'):.4f}" if isinstance(metrics.get('auc_pr'), float) else '-'
        mae = f"{metrics.get('mae_emotion', '-'):.4f}" if isinstance(metrics.get('mae_emotion'), float) else '-'
        r5 = f"{metrics.get('r_at_5', '-'):.4f}" if isinstance(metrics.get('r_at_5'), float) else '-'
        print(f'{model_name:<25} {macro_f1:>10} {auc_pr:>10} {mae:>10} {r5:>10}')
else:
    print('No results file found. Run training first:')
    print('  python src/train.py --task 1 --epochs 20')
    print('  python src/train.py --task 2 --epochs 30')
    print('  python src/train.py --task 3 --epochs 30')
    print('  python src/train.py --task 4 --epochs 30')

print('\nDemo complete!')